# LiGGUF on Kaggle + OpenAI-Compatible Endpoint

This notebook builds [**LiGGUF**](https://github.com/matrixsmaster/ligguf) (`ligguf-fast`) with Qwen3 compatibility fixes, downloads a compatible GGUF model (default: `Qwen/Qwen3-0.6B-GGUF`), verifies local CLI inference, and exposes an **OpenAI-compatible REST API** (`/v1/chat/completions`, `/v1/completions`, `/v1/models`) via FastAPI + Cloudflare Tunnel.

### Kaggle Settings Before Running
- **Internet:** **ON** (Settings -> Internet -> On)
- **Accelerator:** **None** (CPU is sufficient; LiGGUF runs on CPU via OpenMP + AVX2/FMA)

## 1. Configuration

In [ ]:
import os

# Option A: Download from Hugging Face
HF_REPO = "Qwen/Qwen3-0.6B-GGUF"
HF_FILENAME = "Qwen3-0.6B-Q8_0.gguf"

# Option B: Or point directly to a model inside an attached Kaggle Dataset (/kaggle/input/...)
LOCAL_MODEL_PATH = ""  # e.g. "/kaggle/input/my-gguf/model.gguf"

WORK_DIR = "/kaggle/working"
LIGGUF_DIR = f"{WORK_DIR}/ligguf"
MODELS_DIR = f"{WORK_DIR}/models"
CONTEXT_LEN = 2048
PORT = 8000

os.makedirs(MODELS_DIR, exist_ok=True)
print("Configured. CPU count:", os.cpu_count())

## 2. Clone, Patch, and Build `ligguf-fast`
Patches `ligguf` to:
- Size `q` / attention buffers to `n_heads * head_dim` (fixes `SIGABRT` in `rope()` when `n_heads * head_dim != n_embed`, e.g., `Qwen3-0.6B`).
- Fall back to tied embeddings (`t_out = t_embed`) when `output.weight` is omitted.
- Use NeoX split-half RoPE for `qwen3` models.
- Preserve ChatML/Qwen3 special tokens (`<|im_start|>`, `<|im_end|>`, `<think>`, `</think>`) and emit `<<<GEN_START>>>` before generated output.

In [ ]:
import os, subprocess

if not os.path.exists(LIGGUF_DIR):
    subprocess.run(["git", "clone", "https://github.com/matrixsmaster/ligguf.git", LIGGUF_DIR], check=True)
else:
    subprocess.run(["git", "-C", LIGGUF_DIR, "checkout", "--", "fast/"], check=True)

base = f"{LIGGUF_DIR}/fast"

# 1. common.h: add dedicated attention output buffer (n_heads * head_dim)
with open(f"{base}/common.h") as f:
    s = f.read()
s = s.replace(
    "    ftensor xb2; // projected attention output\n",
    "    ftensor xb2; // projected attention output\n    ftensor att_xb; // attention output buffer (n_heads * head_dim)\n"
)
with open(f"{base}/common.h", "w") as f:
    f.write(s)

# 2. common.cpp: size q, att_xb, xq, xq8 using q_dim = n_heads * head_dim (fixes Qwen3-0.6B/1.7B/4B SIGABRT)
with open(f"{base}/common.cpp") as f:
    s = f.read()
s = s.replace(
    "    xq8.resize(m.n_ff / QK8_0);\n    xq.resize(m.n_ff / QK_K);\n",
    "    int q_dim = m.n_heads * m.head_dim;\n    int max_dim = m.n_ff > q_dim ? m.n_ff : q_dim;\n    if (m.n_embed > max_dim) max_dim = m.n_embed;\n    xq8.resize(max_dim / QK8_0);\n    xq.resize(max_dim / QK_K);\n    att_xb.resize(q_dim);\n"
)
s = s.replace("    q.resize(m.n_embed);\n", "    q.resize(q_dim);\n")
with open(f"{base}/common.cpp", "w") as f:
    f.write(s)

# 3. lil_gguf.cpp: support tied embeddings (when output.weight is omitted)
with open(f"{base}/lil_gguf.cpp") as f:
    s = f.read()
s = s.replace(
    "    assert(supported_rowweight(t_embed.type));\n    assert(supported_rowweight(t_out.type));",
    "    if (!t_out.ptr) t_out = t_embed;\n    assert(supported_rowweight(t_embed.type));\n    assert(supported_rowweight(t_out.type));"
)
with open(f"{base}/lil_gguf.cpp", "w") as f:
    f.write(s)

# 4. lil_math.cpp: use NeoX split-half RoPE for qwen3
with open(f"{base}/lil_math.cpp") as f:
    s = f.read()
old_rope = """    for (int h = 0; h < n_heads; h++) {
        float* v = x.data() + h * mdl.head_dim;
        for (int i = 0; i < rope_dim; i += 2) {
            const int m = i >> 1;
            const float ang = pos * rope_freq[m];
            const float c = cosf(ang), s = sinf(ang);
            const float x0 = v[i + 0];
            const float x1 = v[i + 1];
            v[i + 0] = x0 * c - x1 * s;
            v[i + 1] = x0 * s + x1 * c;
        }
    }"""
new_rope = """    const int half = rope_dim >> 1;
    for (int h = 0; h < n_heads; h++) {
        float* v = x.data() + h * mdl.head_dim;
        if (mdl.arch == "qwen3") {
            for (int i = 0; i < half; i++) {
                const float ang = pos * rope_freq[i];
                const float c = cosf(ang), s = sinf(ang);
                const float x0 = v[i];
                const float x1 = v[i + half];
                v[i] = x0 * c - x1 * s;
                v[i + half] = x0 * s + x1 * c;
            }
        } else {
            for (int i = 0; i < rope_dim; i += 2) {
                const int m = i >> 1;
                const float ang = pos * rope_freq[m];
                const float c = cosf(ang), s = sinf(ang);
                const float x0 = v[i + 0];
                const float x1 = v[i + 1];
                v[i + 0] = x0 * c - x1 * s;
                v[i + 1] = x0 * s + x1 * c;
            }
        }
    }"""
assert old_rope in s
s = s.replace(old_rope, new_rope)
with open(f"{base}/lil_math.cpp", "w") as f:
    f.write(s)

# 5. tokenize.cpp: recognize ChatML / Qwen3 special tokens instead of splitting via BPE
with open(f"{base}/tokenize.cpp") as f:
    s = f.read()
old_tok = """vector<int> tokenize(const char* str, bool bos, bool eos)
{
    vector<int> out;
    out.reserve(strlen(str)+2);
    if (bos && g_m.m.add_bos && g_m.m.tok_bos > 0) out.push_back(g_m.m.tok_bos);

    if (g_m.m.merge_rank.empty()) tokenize_bpe_fallback(str,out);
    else tokenize_merge_ranked(str,out);

    if (eos && g_m.m.tok_eos > 0) out.push_back(g_m.m.tok_eos);
    return out;
}"""
new_tok = """static void tokenize_segment(const char* str, vector<int> &out)
{
    if (!*str) return;
    if (g_m.m.merge_rank.empty()) tokenize_bpe_fallback(str,out);
    else tokenize_merge_ranked(str,out);
}

vector<int> tokenize(const char* str, bool bos, bool eos)
{
    vector<int> out;
    out.reserve(strlen(str)+2);
    if (bos && g_m.m.add_bos && g_m.m.tok_bos > 0) out.push_back(g_m.m.tok_bos);

    string input(str);
    size_t pos = 0;
    while (pos < input.size()) {
        size_t best_pos = string::npos;
        size_t best_len = 0;
        const char* specials[] = {"<|im_start|>", "<|im_end|>", "<|endoftext|>", "<think>", "</think>"};
        for (const char* sp : specials) {
            if (!g_m.m.tokens_rev.count(sp)) continue;
            size_t found = input.find(sp, pos);
            if (found != string::npos && (best_pos == string::npos || found < best_pos)) {
                best_pos = found;
                best_len = strlen(sp);
            }
        }
        if (best_pos == string::npos) {
            tokenize_segment(input.c_str() + pos, out);
            break;
        }
        if (best_pos > pos) tokenize_segment(input.substr(pos, best_pos - pos).c_str(), out);
        out.push_back(g_m.m.tokens_rev.at(input.substr(best_pos, best_len)));
        pos = best_pos + best_len;
    }

    if (eos && g_m.m.tok_eos > 0) out.push_back(g_m.m.tok_eos);
    return out;
}"""
assert old_tok in s
s = s.replace(old_tok, new_tok)
with open(f"{base}/tokenize.cpp", "w") as f:
    f.write(s)

# 6. ligguf_fast_qk.cpp: q_dim projection, don't truncate prompt on tok_eos/token 0, clean GEN_START marker
with open(f"{base}/ligguf_fast_qk.cpp") as f:
    s = f.read()
s = s.replace(
    "matmul3(g_m.q,g_m.k,g_m.v,&g_m.xb,&g_m.xq,NULL,g_m.m.tr[l].att_q,g_m.m.tr[l].att_k,g_m.m.tr[l].att_v,g_m.m.n_embed,g_m.m.n_embed,g_m.m.kv_dim,g_m.m.kv_dim);",
    "matmul3(g_m.q,g_m.k,g_m.v,&g_m.xb,&g_m.xq,NULL,g_m.m.tr[l].att_q,g_m.m.tr[l].att_k,g_m.m.tr[l].att_v,g_m.m.n_embed,g_m.m.n_heads * g_m.m.head_dim,g_m.m.kv_dim,g_m.m.kv_dim);"
)
s = s.replace(
    "float* xb = g_m.xb.data() + (h * g_m.m.head_dim);",
    "float* xb = g_m.att_xb.data() + (h * g_m.m.head_dim);"
)
s = s.replace(
    "quantize_q8_K(g_m.xq,g_m.xb);\n        matmul(g_m.xb2,&g_m.xb,&g_m.xq,NULL,g_m.m.tr[l].att_out,g_m.m.n_embed,g_m.m.n_embed);",
    "quantize_q8_K(g_m.xq,g_m.att_xb);\n        matmul(g_m.xb2,&g_m.att_xb,&g_m.xq,NULL,g_m.m.tr[l].att_out,g_m.m.n_heads * g_m.m.head_dim,g_m.m.n_embed);"
)
s = s.replace(
    "    for (int i = 0; i < (int)toks.size(); i++) {\n        int tok = toks.at(i);\n        if (!tok || tok == g_m.m.tok_eos) break;",
    "    for (int i = 0; i < (int)toks.size(); i++) {\n        int tok = toks.at(i);\n        if (tok < 0) break;"
)
s = s.replace(
    "if (!tok || tok == g_m.m.tok_eos) break;",
    "if (tok == g_m.m.tok_eos) break;"
)
s = s.replace(
    "feed_text(g_m.prompt,true,true);",
    "feed_text(g_m.prompt,false,true); printf(\\"\\n<<<GEN_START>>>\\"); fflush(stdout);"
)
with open(f"{base}/ligguf_fast_qk.cpp", "w") as f:
    f.write(s)

subprocess.run(["make", "-C", LIGGUF_DIR, "ligguf-fast"], check=True)
print("Built:", f"{LIGGUF_DIR}/ligguf-fast")

## 3. Download (or Locate) the `.gguf` Model

In [ ]:
from huggingface_hub import hf_hub_download

if LOCAL_MODEL_PATH and os.path.exists(LOCAL_MODEL_PATH):
    MODEL_PATH = LOCAL_MODEL_PATH
else:
    MODEL_PATH = hf_hub_download(
        repo_id=HF_REPO,
        filename=HF_FILENAME,
        local_dir=MODELS_DIR,
    )

os.environ["LIGGUF_MODEL_PATH"] = MODEL_PATH
os.environ["LIGGUF_CTX"] = str(CONTEXT_LEN)
print("Using model:", MODEL_PATH)

## 4. Quick CLI Sanity Check

In [ ]:
import subprocess

prompt = "<|im_start|>user\nExplain what GGUF format is in two sentences.<|im_end|>\n<|im_start|>assistant\n<think>\n\n</think>\n\n"
cmd = [
    f"{LIGGUF_DIR}/ligguf-fast",
    "-m", MODEL_PATH,
    "-c", str(CONTEXT_LEN),
    "-n", "128",
    "-K", "40", "-T", "0.9", "-M", "0.7",
    "-p", prompt,
]

try:
    res = subprocess.run(cmd, capture_output=True, text=True, check=True)
except subprocess.CalledProcessError as e:
    print("STDOUT:\n", e.stdout)
    print("STDERR:\n", e.stderr)
    raise

out = res.stdout.split("<<<GEN_START>>>", 1)[-1].split("\n\nInference-only tok/s:", 1)[0]
print("=== Model Output ===")
print(out.strip())

## 5. Create the OpenAI-Compatible FastAPI Server (`server.py`)

In [ ]:
%%writefile /kaggle/working/server.py
import asyncio
import os
import subprocess
import tempfile
import time
import uuid
from typing import List, Optional

from fastapi import FastAPI, HTTPException
from pydantic import BaseModel

LIGGUF_BIN = "/kaggle/working/ligguf/ligguf-fast"
MODEL_PATH = os.environ.get("LIGGUF_MODEL_PATH", "/kaggle/working/models/Qwen3-0.6B-Q8_0.gguf")
CONTEXT_LEN = int(os.environ.get("LIGGUF_CTX", "2048"))
USE_CHATML = os.environ.get("LIGGUF_CHATML", "1") == "1"

app = FastAPI(title="LiGGUF OpenAI-Compatible API")
gen_lock = asyncio.Lock()  # serialize CPU inference requests


class Message(BaseModel):
    role: str
    content: str


class ChatRequest(BaseModel):
    model: str = "ligguf"
    messages: List[Message]
    max_tokens: Optional[int] = 256
    temperature: Optional[float] = 0.7
    top_p: Optional[float] = 0.9
    top_k: Optional[int] = 40


class CompletionRequest(BaseModel):
    model: str = "ligguf"
    prompt: str
    max_tokens: Optional[int] = 256
    temperature: Optional[float] = 0.7
    top_p: Optional[float] = 0.9
    top_k: Optional[int] = 40


def format_chat(messages: List[Message]) -> str:
    if USE_CHATML:
        parts = [f"<|im_start|>{m.role}\n{m.content}<|im_end|>" for m in messages]
        parts.append("<|im_start|>assistant\n<think>\n\n</think>\n\n")
        return "\n".join(parts)
    lines = [f"{m.role.capitalize()}: {m.content}" for m in messages]
    lines.append("Assistant:")
    return "\n".join(lines)


def run_ligguf(prompt: str, max_tokens: int, temperature: float, top_p: float, top_k: int) -> str:
    with tempfile.NamedTemporaryFile("w", suffix=".txt", delete=False) as tf:
        tf.write(prompt)
        prompt_file = tf.name

    try:
        cmd = [
            LIGGUF_BIN,
            "-m", MODEL_PATH,
            "-c", str(CONTEXT_LEN),
            "-f", prompt_file,
            "-n", str(max_tokens),
            "-K", str(top_k),
            "-T", str(top_p),
            "-M", str(max(temperature, 0.0)),
        ]
        if temperature <= 0:
            cmd.append("-G")

        res = subprocess.run(cmd, capture_output=True, text=True, check=True)
        out = res.stdout

        if "<<<GEN_START>>>" in out:
            out = out.split("<<<GEN_START>>>", 1)[1]
        if "\n\nInference-only tok/s:" in out:
            out = out.split("\n\nInference-only tok/s:", 1)[0]

        for stop_tok in ("<|im_end|>", "<|endoftext|>"):
            if stop_tok in out:
                out = out.split(stop_tok, 1)[0]

        return out.strip()
    finally:
        if os.path.exists(prompt_file):
            os.remove(prompt_file)


@app.get("/health")
async def health():
    return {"status": "ok", "model": MODEL_PATH}


@app.get("/v1/models")
async def list_models():
    return {
        "object": "list",
        "data": [{"id": "ligguf", "object": "model", "owned_by": "local"}],
    }


@app.post("/v1/chat/completions")
async def chat_completions(req: ChatRequest):
    prompt = format_chat(req.messages)
    async with gen_lock:
        try:
            text = await asyncio.to_thread(
                run_ligguf,
                prompt,
                req.max_tokens or 256,
                req.temperature if req.temperature is not None else 0.7,
                req.top_p if req.top_p is not None else 0.9,
                req.top_k if req.top_k is not None else 40,
            )
        except subprocess.CalledProcessError as e:
            raise HTTPException(status_code=500, detail=e.stderr or str(e))

    return {
        "id": f"chatcmpl-{uuid.uuid4().hex[:12]}",
        "object": "chat.completion",
        "created": int(time.time()),
        "model": req.model,
        "choices": [
            {
                "index": 0,
                "message": {"role": "assistant", "content": text},
                "finish_reason": "stop",
            }
        ],
    }


@app.post("/v1/completions")
async def completions(req: CompletionRequest):
    async with gen_lock:
        try:
            text = await asyncio.to_thread(
                run_ligguf,
                req.prompt,
                req.max_tokens or 256,
                req.temperature if req.temperature is not None else 0.7,
                req.top_p if req.top_p is not None else 0.9,
                req.top_k if req.top_k is not None else 40,
            )
        except subprocess.CalledProcessError as e:
            raise HTTPException(status_code=500, detail=e.stderr or str(e))

    return {
        "id": f"cmpl-{uuid.uuid4().hex[:12]}",
        "object": "text_completion",
        "created": int(time.time()),
        "model": req.model,
        "choices": [{"index": 0, "text": text, "finish_reason": "stop"}],
    }

## 6. Start the Local API Server & Test Locally

In [ ]:
import subprocess, time, urllib.request

!pip install -q fastapi uvicorn openai

subprocess.run(["pkill", "-f", "uvicorn server:app"], check=False)
time.sleep(0.5)

server_proc = subprocess.Popen(
    ["uvicorn", "server:app", "--host", "0.0.0.0", "--port", str(PORT)],
    cwd=WORK_DIR,
    stdout=open(f"{WORK_DIR}/api.log", "w"),
    stderr=subprocess.STDOUT,
    env=os.environ.copy(),
)

for _ in range(20):
    try:
        with urllib.request.urlopen(f"http://127.0.0.1:{PORT}/health") as r:
            print("Server healthy:", r.read().decode())
            break
    except Exception:
        time.sleep(0.5)

In [ ]:
from openai import OpenAI

client = OpenAI(base_url=f"http://127.0.0.1:{PORT}/v1", api_key="not-needed")
resp = client.chat.completions.create(
    model="ligguf",
    messages=[{"role": "user", "content": "Say hello and give 1 fun fact about C++."}],
    max_tokens=100,
    temperature=0.7,
)
print(resp.choices[0].message.content)

## 7. Expose Publicly via Cloudflare Tunnel
Run this cell to get a public `https://*.trycloudflare.com` URL. Keep this cell running while external clients call the endpoint (use `/v1` as `base_url` in the OpenAI SDK).

In [ ]:
%%bash
if [ ! -f /usr/local/bin/cloudflared ]; then
    wget -q https://github.com/cloudflare/cloudflared/releases/latest/download/cloudflared-linux-amd64 -O /usr/local/bin/cloudflared
    chmod +x /usr/local/bin/cloudflared
fi

cloudflared tunnel --url http://localhost:8000